## Data Cleaning

### Data Dictionary:

- **origin**: IATA code of departure airport
- **destination**: IATA code of arrival airport
- **query_date**: (YYYY-MM-DD) The date the flight was queried for
- **departure_time**: (YYYY-MM-DDTHH:MM) Scheduled departure datetime of the flight
- **arrival_time**: (YYYY-MM-DDTHH:MM) Scheduled arrival datetime of the flight
- **day_of_week**: Day of the week of departure (Monday=0, Sunday=6)
- **month**: Month of departure
- **days_until_departure**: Number of days between query date and flight departure
- **airline**: IATA carrier code of airline
- **aircraft**: Aircraft code (type)
- **trip_duration_minutes**: Total flight duration in minutes
- **number_of_stops**: Number of stops/layovers
- **currency**: Currency code of flight price
- **base_price**: Base fare of the flight
- **total_price**: Total fare including taxes & fees
- **bookable_seats**: Number of seats currently available for booking
- **checked_bags**: Number of included checked bags



Here are the step by step processes to clean and prepare our datsets for Exploratory Data Analysis (EDA).
- Import pandas library
- Load flight data from Excel sheet `flights_dataset_updated_v2.xlsx` (`flights_dataset` sheet)
- Inspect the dataset with `head()`, `info()`, `describe()`
- Check for duplicates with `df[df.duplicated()]`
- Drop unnecessary columns: `checked_bags`, `cabin_bags`
- Convert `departure_time` and `arrival_time` to datetime and split into date and clock time columns:
    - `departure_date`, `departure_clock_time`
    - `arrival_date`, `arrival_clock_time`
- Recalculate `days_until_departure` as `departure_date - query_date`
- Rearrange columns and rename:
    - `day_of_week` → `day_of_week_departure`
    - `month` → `month_departure`
- Load airport location data from the `Airports` sheet
- Merge airport coordinates into the flight data for origin and destination
- Define a Haversine distance function and calculate `distance_km` for each route
- Drop redundant columns to leave the cleaned dataset
- Export the cleaned dataset to `flights_cleaned_df.csv` for EDA

Import pandas library

In [56]:
# Import libraries
import pandas as pd

from math import radians, cos, sin, asin, sqrt

#### Reading the data
Read data from an Excel sheet

In [57]:
# extract the 'flights_dataset' sheet from the excel file
df = pd.read_excel('./flights_dataset_updated_v2.xlsx','flights_dataset')

Inspect the dataset with `head()`, `info()`, `describe()`

In [58]:
df.head(3)

,origin,destination,query_date,departure_time,arrival_time,day_of_week,month,days_until_departure,airline,aircraft,trip_duration_minutes,number_of_stops,currency,base_price,total_price,bookable_seats,checked_bags,cabin_bags
0,YOW,YYZ,2026-03-08,2026-03-09T05:40:00,2026-03-09T07:01:00,0,3,0,WS,7M8,81,0,EUR,216,287.58,7,0,0
1,YOW,YYZ,2026-03-08,2026-03-09T06:45:00,2026-03-09T08:00:00,0,3,0,PD,295,75,0,EUR,216,290.50,9,0,0
2,YOW,YYZ,2026-03-08,2026-03-09T10:00:00,2026-03-09T11:15:00,0,3,0,PD,295,75,0,EUR,216,290.50,9,0,0


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43242 entries, 0 to 43241
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   origin                 43242 non-null  object        
 1   destination            43242 non-null  object        
 2   query_date             43242 non-null  datetime64[ns]
 3   departure_time         43242 non-null  object        
 4   arrival_time           43242 non-null  object        
 5   day_of_week            43242 non-null  int64         
 6   month                  43242 non-null  int64         
 7   days_until_departure   43242 non-null  int64         
 8   airline                43242 non-null  object        
 9   aircraft               43242 non-null  object        
 10  trip_duration_minutes  43242 non-null  int64         
 11  number_of_stops        43242 non-null  int64         
 12  currency               43242 non-null  object        
 13  b

In [60]:
df.describe()

,query_date,day_of_week,month,days_until_departure,trip_duration_minutes,number_of_stops,base_price,total_price,bookable_seats,checked_bags,cabin_bags
count,43242,43242.000000,43242.000000,43242.0,43242.000000,43242.000000,43242.000000,43242.000000,43242.000000,43242.0,43242.0
mean,2026-03-07 23:59:59.999999744,4.749248,6.177466,0.0,291.460825,0.777670,199.783498,266.358800,7.859812,0.0,0.0
min,2026-03-08 00:00:00,0.000000,1.000000,0.0,64.000000,0.000000,22.000000,46.530000,1.000000,0.0,0.0
25%,2026-03-08 00:00:00,4.000000,5.000000,0.0,229.000000,1.000000,130.000000,187.937500,7.000000,0.0,0.0
50%,2026-03-08 00:00:00,6.000000,6.000000,0.0,325.000000,1.000000,168.000000,231.470000,9.000000,0.0,0.0
75%,2026-03-08 00:00:00,6.000000,8.000000,0.0,367.000000,1.000000,224.000000,300.890000,9.000000,0.0,0.0
max,2026-03-08 00:00:00,6.000000,12.000000,0.0,717.000000,2.000000,2432.000000,2789.940000,9.000000,0.0,0.0
std,NaN,1.961507,2.103253,0.0,118.209078,0.425112,133.289708,148.262115,1.642531,0.0,0.0


Check for duplicates with `df[df.duplicated()]`

In [61]:
# investigate duplicates
df[df.duplicated()].head()

,origin,destination,query_date,departure_time,arrival_time,day_of_week,month,days_until_departure,airline,aircraft,trip_duration_minutes,number_of_stops,currency,base_price,total_price,bookable_seats,checked_bags,cabin_bags
18027,YOW,YVR,2026-03-08,2026-10-25T13:05:00,2026-10-25T21:50:00,6,10,0,WS,73W,395,2,EUR,106,173.25,9,0,0
18031,YOW,YVR,2026-03-08,2026-10-25T05:55:00,2026-10-25T12:35:00,6,10,0,WS,7M8,424,2,EUR,106,180.38,9,0,0
18041,YOW,YVR,2026-03-08,2026-10-25T06:00:00,2026-10-25T16:10:00,6,10,0,WS,7M8,420,2,EUR,106,199.75,9,0,0
18056,YOW,YVR,2026-03-08,2026-10-25T05:55:00,2026-10-25T14:01:00,6,10,0,WS,7M8,415,2,EUR,218,321.92,9,0,0
18057,YOW,YVR,2026-03-08,2026-10-25T05:55:00,2026-10-25T14:01:00,6,10,0,WS,7M8,415,2,EUR,218,321.92,9,0,0


We can conclude we have no duplicates as each row returned are unique

Convert `departure_time` and `arrival_time` to datetime and split into date and clock time columns:
- `departure_date`, `departure_clock_time`
- `arrival_date`, `arrival_clock_time`

In [62]:
# convert to datetime format
df['departure_time'] = pd.to_datetime(df['departure_time'])
df['arrival_time'] = pd.to_datetime(df['arrival_time'])

# extarct date and time from the datetime columns
df['departure_date'] = df['departure_time'].dt.date
df['departure_clock_time'] = df['departure_time'].dt.time
df['arrival_date'] = df['arrival_time'].dt.date
df['arrival_clock_time'] = df['arrival_time'].dt.time

Recalculate `days_until_departure` as `departure_date - query_date`

In [63]:
# convert to datatime format
df['query_date'] = pd.to_datetime(df['query_date'])
df['departure_date'] = pd.to_datetime(df['departure_date'])

# get the number of days until departure
df['days_until_departure'] = (df['departure_date'] - df['query_date']).dt.days

Rearrange columns and rename:
- `day_of_week` → `day_of_week_departure`
- `month` → `month_departure`

In [64]:
# rearrage columns
df = df.loc[:,
       ['origin', 'destination', 'airline', 'aircraft', 'query_date'
        ,'departure_date','departure_clock_time', 'day_of_week','month', 'arrival_date','arrival_clock_time'
        , 'days_until_departure','trip_duration_minutes','number_of_stops',
        'bookable_seats','base_price','total_price']]

# rename columns for clarity
df.rename(columns = {'day_of_week' : 'day_of_week_departure',
                     'month' : 'month_departure'}, inplace = True)

In [65]:
# preview the cleaned dataset
df.sample(5)

,origin,destination,airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,arrival_clock_time,days_until_departure,trip_duration_minutes,number_of_stops,bookable_seats,base_price,total_price
38461,YYC,YYZ,WS,73W,2026-03-08,2026-07-05,20:15:00,6,7,2026-07-06,05:50:00,119,280,1,7,110,165.19
31895,YVR,YYZ,AC,320,2026-03-08,2026-08-23,22:00:00,6,8,2026-08-24,11:12:00,168,315,1,9,220,264.79
42297,YYC,YVR,WS,73W,2026-03-08,2026-08-16,23:30:00,6,8,2026-08-17,08:55:00,161,155,1,7,85,152.86
31004,YVR,YYZ,AC,321,2026-03-08,2026-07-05,10:10:00,6,7,2026-07-05,17:45:00,119,275,0,9,245,273.81
41363,YYC,YVR,WS,73W,2026-03-08,2026-06-21,23:35:00,6,6,2026-06-22,00:01:00,105,86,0,7,65,108.67


### Data Preparation

In [66]:
# import the airports sheet
df2 = pd.read_excel('./flights_dataset_updated_v2.xlsx','Airports')

# import the airlines sheet
df4 = pd.read_excel('./flights_dataset_updated_v2.xlsx','Airlines')

df4.head()

,ID,Airline,Name
0,1,WS,WestJet
1,2,PD,Porter Airlines Canada Limited
2,3,AC,Air Canada
3,4,TS,Air Transat
4,5,5T,First Air Canadian North


In [67]:
# aiport name and city mapping along with geographical coordinates
df2.head()

,Code,City,Name,x,y
0,YOW,Ottawa,Ottawa MacDonald Cartier Intl,45.3233,-75.6667
1,YVR,Vancouver,Vancouver International,49.1950,-123.1833
2,YYC,Calgary,Calgary International,51.1133,-114.0200
3,YYZ,Toronto,Toronto Lester B Pearson International,43.6767,-79.6300


Merge airport coordinates into the flight data for origin and destination

In [68]:
df3 = df.copy()

# join tables
df3 = df3.merge(df2, left_on='origin', right_on='Code', how='left', suffixes=('', '_origin'))
df3 = df3.merge(df2, left_on='destination', right_on='Code', how='left', suffixes=('', '_destination'))

# add the airline names to the main dataframe
df3 = df3.merge(df4, left_on='airline', right_on='Airline', how='left', suffixes=('', '_airline'))

#previw data
df3.sample(3)

,origin,destination,airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,...,x,y,Code_destination,City_destination,Name_destination,x_destination,y_destination,ID,Airline,Name_airline
42488,YYC,YVR,WS,7M8,2026-03-08,2026-08-30,10:00:00,6,8,2026-08-30,...,51.1133,-114.0200,YVR,Vancouver,Vancouver International,49.195,-123.1833,1,WS,WestJet
42195,YYC,YVR,WS,73W,2026-03-08,2026-08-09,23:40:00,6,8,2026-08-10,...,51.1133,-114.0200,YVR,Vancouver,Vancouver International,49.195,-123.1833,1,WS,WestJet
8965,YOW,YVR,AC,CR9,2026-03-08,2026-05-30,16:00:00,5,5,2026-05-30,...,45.3233,-75.6667,YVR,Vancouver,Vancouver International,49.195,-123.1833,3,AC,Air Canada


Drop redundant columns to leave the cleaned dataset

In [69]:
# Select columns for the final dataset
df3 = df3.loc[:,['origin','City','destination','City_destination','Name_airline', 'aircraft', 'query_date',
       'departure_date', 'departure_clock_time', 'day_of_week_departure',
       'month_departure', 'arrival_date', 'arrival_clock_time',
       'days_until_departure', 'trip_duration_minutes', 'number_of_stops',
       'bookable_seats', 'base_price']]

df3.sample(5)

,origin,City,destination,City_destination,Name_airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,arrival_clock_time,days_until_departure,trip_duration_minutes,number_of_stops,bookable_seats,base_price
30424,YVR,Vancouver,YYZ,Toronto,Air Canada,320,2026-03-08,2026-05-31,10:45:00,6,5,2026-05-31,23:49:00,84,317,1,9,171
19195,YOW,Ottawa,YYC,Calgary,Air Canada,223,2026-03-08,2026-08-02,09:20:00,6,8,2026-08-02,14:15:00,147,324,1,9,220
20161,YYZ,Toronto,YOW,Ottawa,Air Canada,788,2026-03-08,2026-07-05,16:30:00,6,7,2026-07-05,20:15:00,119,126,1,8,383
11743,YOW,Ottawa,YVR,Vancouver,Air Canada,223,2026-03-08,2026-07-18,15:55:00,5,7,2026-07-18,22:04:00,132,373,1,9,215
34605,YVR,Vancouver,YYC,Calgary,WestJet,7M8,2026-03-08,2026-08-02,07:00:00,6,8,2026-08-02,09:30:00,147,90,0,7,85


### Export data for EDA

In [70]:
df3.to_csv('./flights_cleaned_df.csv', index=False)
print('File Saved')

File Saved
